# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 148MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=702626a4-beed-46d8-857b-55587e8a0988
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 213MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 159MB/s]


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [5]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L-infinity PGD (use input_grad and clip_box)
    delta = torch.empty_like(x).uniform_(-eps, eps) if rand_init else torch.zeros_like(x)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = (delta + alpha * grad.sign()).clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L2 PGD (use input_grad, l2_norm and clip_box)
    delta = torch.randn_like(x) if rand_init else torch.zeros_like(x)
    if rand_init:
        radius = torch.rand(x.size(0), 1, 1, 1, device=x.device).pow(1.0 / delta[0].numel())
        delta = delta / l2_norm(delta).clamp_min(1e-12) * (eps * radius)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)
        delta = delta * (eps / l2_norm(delta).clamp_min(1e-12)).clamp(max=1.0)
        delta = clip_box(x, delta)
    return (x + delta).detach()

In [6]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [7]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [8]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.5  l2 60.3  l1 23.2  union 23.0   (12 s)


{'clean': 78.7, 'linf': 48.5, 'l2': 60.3, 'l1': 23.2, 'union': 23.0}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [9]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [10]:
def single_norm_loss(model, x, y, norm):
    # TODO: put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [11]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.200 (6 s)
epoch 2: loss 1.180 (12 s)
epoch 3: loss 1.154 (17 s)
epoch 4: loss 1.132 (23 s)
epoch 5: loss 1.098 (29 s)
epoch 6: loss 1.069 (34 s)
epoch 7: loss 1.043 (40 s)
epoch 8: loss 1.027 (46 s)
epoch 9: loss 1.011 (51 s)
epoch 10: loss 1.004 (57 s)
clean 79.4  linf 44.6  l2 57.6  l1 19.5  union 19.3   (9 s)


{'clean': 79.4, 'linf': 44.6, 'l2': 57.6, 'l1': 19.5, 'union': 19.3}

In [12]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.890 (6 s)
epoch 2: loss 0.823 (11 s)
epoch 3: loss 0.778 (17 s)
epoch 4: loss 0.737 (23 s)
epoch 5: loss 0.704 (29 s)
epoch 6: loss 0.665 (35 s)
epoch 7: loss 0.622 (40 s)
epoch 8: loss 0.598 (46 s)
epoch 9: loss 0.580 (52 s)
epoch 10: loss 0.572 (58 s)
clean 81.8  linf 32.7  l2 60.7  l1 28.8  union 25.0   (9 s)


{'clean': 81.8, 'linf': 32.7, 'l2': 60.7, 'l1': 28.8, 'union': 25.0}

In [13]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.231 (7 s)
epoch 2: loss 0.847 (13 s)
epoch 3: loss 0.547 (20 s)
epoch 4: loss 0.441 (27 s)
epoch 5: loss 0.346 (33 s)
epoch 6: loss 0.273 (40 s)
epoch 7: loss 0.230 (47 s)
epoch 8: loss 0.198 (53 s)
epoch 9: loss 0.174 (60 s)
epoch 10: loss 0.170 (67 s)
clean 84.8  linf 0.0  l2 0.0  l1 26.3  union 0.0   (9 s)


{'clean': 84.8, 'linf': 0.0, 'l2': 0.0, 'l1': 26.3, 'union': 0.0}

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [14]:
def ramp_loss(model, x, y, lam=0.5):
    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()

    logits_q = model(x_q)
    logits_r = model(x_r)
    loss_q = F.cross_entropy(logits_q, y, reduction="none")
    loss_r = F.cross_entropy(logits_r, y, reduction="none")
    adv_loss = torch.maximum(loss_q, loss_r).mean()
    correct_q = logits_q.detach().argmax(dim=1).eq(y)
    if correct_q.any():
        kl_loss = F.kl_div(
            F.log_softmax(logits_r[correct_q], dim=1),
            F.softmax(logits_q[correct_q], dim=1),
            reduction="batchmean",
        )
    else:
        kl_loss = logits_q.sum() * 0.0
    return adv_loss + lam * kl_loss


def gradient_projection(g_n, g_a):
    # TODO: GP(g_n, g_a) for one parameter tensor
    dot = torch.sum(g_n * g_a)
    norm_product = torch.linalg.vector_norm(g_n) * torch.linalg.vector_norm(g_a)
    cos = dot / norm_product.clamp_min(1e-12)
    return cos.clamp(min=0.0) * g_n

In [15]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.430 (13 s)
epoch 2: loss 1.335 (27 s)
epoch 3: loss 1.300 (40 s)
epoch 4: loss 1.268 (54 s)
epoch 5: loss 1.253 (67 s)
epoch 6: loss 1.223 (80 s)
epoch 7: loss 1.199 (94 s)
epoch 8: loss 1.182 (107 s)
epoch 9: loss 1.174 (120 s)
epoch 10: loss 1.166 (134 s)
clean 78.6  linf 42.3  l2 62.5  l1 45.0  union 40.0   (9 s)


{'clean': 78.6, 'linf': 42.3, 'l2': 62.5, 'l1': 45.0, 'union': 40.0}

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [16]:
def my_loss(model, x, y, ablation=False):
    norms = ("linf", "l2", "l1")
    if not hasattr(model, "norm_failure"):
        model.norm_failure = torch.ones(3, device=x.device)
    scores = model.norm_failure
    probs = torch.ones_like(scores) / 3 if ablation else 0.2 / 3 + 0.8 * scores / scores.sum()
    k = torch.multinomial(probs, 1).item()

    model.eval()
    x_adv = attack(model, x, y, norms[k])
    model.train()
    logits = model(x_adv)
    with torch.no_grad():
        failure = (logits.argmax(1) != y).float().mean()
        scores[k] = 0.9 * scores[k] + 0.1 * failure
    return F.cross_entropy(logits, y)


In [20]:
model_mine = adv_train(load_base_model(), my_loss)
test_model(model_mine)

epoch 1: loss 1.211 (6 s)
epoch 2: loss 1.146 (12 s)
epoch 3: loss 1.133 (19 s)
epoch 4: loss 1.096 (25 s)
epoch 5: loss 1.048 (31 s)
epoch 6: loss 0.993 (37 s)
epoch 7: loss 1.004 (43 s)
epoch 8: loss 0.979 (50 s)
epoch 9: loss 0.952 (56 s)
epoch 10: loss 0.967 (62 s)
clean 78.0  linf 41.0  l2 62.5  l1 48.0  union 39.9   (9 s)


{'clean': 78.0, 'linf': 41.0, 'l2': 62.5, 'l1': 48.0, 'union': 39.9}

In [18]:
import pandas as pd


def run_experiment(name, loss_fn, seed=0, **kwargs):
    start = time.time()
    model = adv_train(load_base_model(), loss_fn, seed=seed, **kwargs)
    seconds = time.time() - start
    acc = test_model(model)
    return model, {"method": name, "seed": seed, "time_s": round(seconds, 1), **acc}

rows = [{"method": "Base", "seed": "-", "time_s": 0.0, **test_model(load_base_model())}]
my_models = {}
for name, loss_fn, kwargs in [
    ("My Method", my_loss, {}),
    ("RAMP", ramp_loss, {"gp_beta": 0.5}),
]:
    for seed in (0, 1):
        model, record = run_experiment(name, loss_fn, seed, **kwargs)
        rows.append(record)
        if name == "My Method":
            my_models[seed] = model

_, ablation_record = run_experiment("Ablation (uniform)", my_loss, 0, ablation=True)
rows.append(ablation_record)
results = pd.DataFrame(rows)
display(results)

metrics = ["clean", "linf", "l2", "l1", "union", "time_s"]
means = results[results.method.isin(["My Method", "RAMP"])].groupby("method")[metrics].mean()
print("Two-seed means (My Method and RAMP):")
display(means.round(2))

clean 78.7  linf 48.5  l2 60.3  l1 23.2  union 23.0   (9 s)
epoch 1: loss 1.210 (6 s)
epoch 2: loss 1.147 (12 s)
epoch 3: loss 1.133 (18 s)
epoch 4: loss 1.096 (24 s)
epoch 5: loss 1.047 (30 s)
epoch 6: loss 0.993 (36 s)
epoch 7: loss 1.003 (42 s)
epoch 8: loss 0.979 (48 s)
epoch 9: loss 0.952 (54 s)
epoch 10: loss 0.969 (60 s)
clean 78.0  linf 41.2  l2 62.2  l1 48.1  union 39.9   (9 s)
epoch 1: loss 1.212 (6 s)
epoch 2: loss 1.155 (12 s)
epoch 3: loss 1.102 (18 s)
epoch 4: loss 1.079 (24 s)
epoch 5: loss 1.055 (30 s)
epoch 6: loss 1.022 (36 s)
epoch 7: loss 0.968 (42 s)
epoch 8: loss 0.985 (49 s)
epoch 9: loss 0.958 (55 s)
epoch 10: loss 0.958 (61 s)
clean 78.0  linf 41.0  l2 62.8  l1 48.5  union 39.9   (9 s)
epoch 1: loss 1.429 (13 s)
epoch 2: loss 1.335 (27 s)
epoch 3: loss 1.299 (40 s)
epoch 4: loss 1.268 (54 s)
epoch 5: loss 1.253 (67 s)
epoch 6: loss 1.223 (81 s)
epoch 7: loss 1.198 (94 s)
epoch 8: loss 1.182 (108 s)
epoch 9: loss 1.174 (121 s)
epoch 10: loss 1.165 (135 s)
clean 

,method,seed,time_s,clean,linf,l2,l1,union
0,Base,-,0.0,78.7,48.5,60.3,23.2,23.0
1,My Method,0,60.5,78.0,41.2,62.2,48.1,39.9
2,My Method,1,60.7,78.0,41.0,62.8,48.5,39.9
3,RAMP,0,134.6,78.8,42.4,62.3,45.0,40.1
4,RAMP,1,134.0,78.5,42.3,62.2,45.9,40.7
5,Ablation (uniform),0,61.5,78.8,40.4,62.5,47.6,39.5


Two-seed means (My Method and RAMP):


,clean,linf,l2,l1,union,time_s
method,,,,,,
My Method,78.00,41.10,62.50,48.30,39.9,60.6
RAMP,78.65,42.35,62.25,45.45,40.4,134.3


In [19]:
best_seed = max((0, 1), key=lambda s: float(results[(results.method == "My Method") & (results.seed == s)].union.iloc[0]))
print("Best My Method seed:", best_seed)
strong = test_model(my_models[best_seed], steps=200, restarts=5)
default = results[(results.method == "My Method") & (results.seed == best_seed)].iloc[0]
display(pd.DataFrame([default[metrics[:-1]].to_dict(), strong], index=["Default", "200 steps / 5 restarts"]))

if means.loc["My Method", "time_s"] > means.loc["RAMP", "time_s"]:
    extra_epochs = max(11, round(10 * means.loc["My Method", "time_s"] / means.loc["RAMP", "time_s"]))
    print("My Method is slower. Compare with RAMP trained for", extra_epochs, "epochs.")
    matched = [run_experiment("RAMP (time-matched)", ramp_loss, seed, epochs=extra_epochs, gp_beta=0.5)[1]
               for seed in (0, 1)]
    display(pd.DataFrame(matched))

Best My Method seed: 0
clean 78.0  linf 41.1  l2 62.2  l1 46.4  union 39.6   (93 s)


,clean,linf,l2,l1,union
Default,78.0,41.2,62.2,48.1,39.9
200 steps / 5 restarts,78.0,41.1,62.2,46.4,39.6
